In [1]:
# load Libraries
import os
from pathlib import Path
from dotenv import load_dotenv  # type: ignore
from kaggle.api.kaggle_api_extended import KaggleApi  # type: ignore
import re
from collections import Counter
import pickle
from pathlib import Path
import numpy as np

In [2]:
# Explicitly point to the .env file relative to this notebook
env_path = Path(__file__).parent / '.env' if '__file__' in dir() else Path('.env')
load_dotenv(dotenv_path=env_path)

os.environ['KAGGLE_USERNAME'] = os.getenv('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = os.getenv('KAGGLE_KEY')

In [3]:
# Create the 'raw' directory if it doesn't exist
Path("raw").mkdir(exist_ok=True)

api = KaggleApi()
api.authenticate()

api.dataset_download_files(
    "yorkyong/text8-zip",
    path="raw",
    unzip=True
)

Dataset URL: https://www.kaggle.com/datasets/yorkyong/text8-zip


In [4]:
# Read the text8 dataset and tokenize it
with open("raw/text8", "r", encoding="utf-8") as f:
    text = f.read()

# Preprocess the text: convert to lowercase and remove punctuation
text = text.lower()
text = re.sub(r"[^a-z\s]", "", text) 

tokens = text.split()
print(len(tokens))
print(tokens[:20])

17005207
['anarchism', 'originated', 'as', 'a', 'term', 'of', 'abuse', 'first', 'used', 'against', 'early', 'working', 'class', 'radicals', 'including', 'the', 'diggers', 'of', 'the', 'english']


In [5]:
# Analyze word frequencies
word_counts = Counter(tokens)

print("Total tokens:", len(tokens))
print("Total unique words:", len(word_counts))
print("Top 10 words:", word_counts.most_common(10))

Total tokens: 17005207
Total unique words: 253854
Top 10 words: [('the', 1061396), ('of', 593677), ('and', 416629), ('one', 411764), ('in', 372201), ('a', 325873), ('to', 316376), ('zero', 264975), ('nine', 250430), ('two', 192644)]


In [6]:
# Filter out infrequent words
min_count = 5
filtered_word_counts = {w: c for w, c in word_counts.items() if c >= min_count}

In [7]:
word2idx = {w: i for i, w in enumerate(filtered_word_counts.keys())}
idx2word = {i: w for w, i in word2idx.items()}

vocab_size = len(word2idx)
print("New Vocab Size:", vocab_size)

# Convert the original 17 million tokens into IDs, keeping the exact sentence structure!
# We use a set for faster lookup
valid_words = set(word2idx.keys())
indexed_tokens = [word2idx[w] for w in tokens if w in valid_words]

print("Total indexed tokens:", len(indexed_tokens))

New Vocab Size: 71290
Total indexed tokens: 16718844


In [8]:
# Create training pairs for the skip-gram model
window_size = 2
pairs = []

for i in range(window_size, len(indexed_tokens) - window_size):
    target = indexed_tokens[i]

    for j in range(-window_size, window_size + 1):
        if j != 0:
            context = indexed_tokens[i + j]
            pairs.append((target, context))

print(len(pairs))

66875360


In [9]:
# 1. Calculate word frequencies to the power of 0.75 for Negative Sampling
total_count = sum(filtered_word_counts.values())
freqs = [filtered_word_counts[idx2word[i]] / total_count for i in range(vocab_size)]
unigram_dist = np.array(freqs) ** 0.75
unigram_dist = unigram_dist / unigram_dist.sum() # Normalize to make it a valid probability

# 2. Save all processed data
Path("processed").mkdir(exist_ok=True)

with open("processed/word2idx.pkl", "wb") as f:
    pickle.dump(word2idx, f)

with open("processed/idx2word.pkl", "wb") as f:
    pickle.dump(idx2word, f)

with open("processed/indexed_tokens.pkl", "wb") as f:
    pickle.dump(indexed_tokens, f)

with open("processed/pairs.pkl", "wb") as f:
    pickle.dump(pairs, f)

with open("processed/unigram_dist.pkl", "wb") as f:
    pickle.dump(unigram_dist, f)

print("Preprocessing complete. All 5 files saved to 'processed' directory.")

Preprocessing complete. All 5 files saved to 'processed' directory.
